## QuickStart: Compare Runs, Choose an model and deplot it a RestAPi

In [1]:
import keras
import numpy as np
import pandas as pd

from hyperopt import STATUS_OK, Trials, fmin, hp, tpe

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

from tensorflow.keras.layers import Dense, Input, Normalization
from tensorflow.keras.models import Sequential

c:\Users\singh\OneDrive\Desktop\ANN with MLflow\AnnEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [3]:
# ── Load & split data ──────────────────────────────────────────────────────────
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)

train, test = train_test_split(data, test_size=0.25, random_state=42)

train_x = train.drop(columns=["quality"]).values
train_y = train["quality"].values.ravel()
test_x  = test.drop(columns=["quality"]).values
test_y  = test["quality"].values.ravel()

train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)

signature = infer_signature(train_x, train_y)

In [4]:
# ── Model training ─────────────────────────────────────────────────────────────
def train_model(params, epochs, train_x, train_y, valid_x, valid_y, parent_run_id):
    norm_layer = Normalization()
    norm_layer.adapt(train_x)

    model = Sequential([
        Input(shape=(train_x.shape[1],)),
        norm_layer,
        Dense(64, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"],
            momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()]
    )

    # ✅ Pass parent_run_id explicitly so nested runs attach correctly
    with mlflow.start_run(nested=True, parent_run_id=parent_run_id):
        model.fit(
            train_x, train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
            verbose=0
        )

        eval_result = model.evaluate(valid_x, valid_y, batch_size=64, verbose=0)
        eval_rmse = eval_result[1]

        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

    return {"loss": eval_rmse, "status": STATUS_OK}

In [13]:
# ── Experiment ─────────────────────────────────────────────────────────────────
mlflow.set_experiment("/wine-quality")

with mlflow.start_run(run_name="hyperopt-search") as parent_run:
    parent_run_id = parent_run.info.run_id  # ✅ Capture parent run ID

    def objective(params):
        return train_model(
            params,
            epochs=3,
            train_x=train_x,
            train_y=train_y,
            valid_x=valid_x,
            valid_y=valid_y,
            parent_run_id=parent_run_id  # ✅ Pass it into each trial
        )

    space = {
        "lr":       hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
        "momentum": hp.uniform("momentum", 0.0, 1.0)
    }

    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # ✅ Log best params/metric to the parent run
    mlflow.log_params(best)
    mlflow.log_metric("best_eval_rmse", best_run["loss"])

    print(f"Best Parameters: {best}")
    print(f"Best Eval RMSE:  {best_run['loss']}")

🏃 View run indecisive-hawk-123 at: http://127.0.0.1:5000/#/experiments/2/runs/87cf745a29184898ac4ce3bb34374701

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

🏃 View run useful-sow-730 at: http://127.0.0.1:5000/#/experiments/2/runs/f1cc227042374a7ab046c5599002b9b8

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run wise-robin-566 at: http://127.0.0.1:5000/#/experiments/2/runs/736cee976e0e4164ac786a189091492a

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run orderly-hound-148 at: http://127.0.0.1:5000/#/experiments/2/runs/599f006337b04cbebb0b3872f235a278

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

100%|██████████| 4/4 [00:03<00:00,  1.00trial/s, best loss: 0.7305884957313538]
Best Parameters: {'lr': np.float64(0.019169398086378808), 'momentum': np.float64(0.8445776728026438)}
Best Eval RMSE:  0.7305884957313538
🏃 View run hyperopt-search at: http://127.0.0

In [11]:
trials.results

[{'loss': 1.4224812984466553, 'status': 'ok'},
 {'loss': 5.27094030380249, 'status': 'ok'},
 {'loss': 0.7725412845611572, 'status': 'ok'},
 {'loss': 5.435359001159668, 'status': 'ok'}]